In [51]:
import pandas as pd
from scipy import stats

Basic anti duplications

In [52]:
# Load forms data
df = pd.read_csv('raw/Encuesta TFG.csv')



df = df.drop_duplicates(subset=['Número de participante'], keep='first').copy()
df['Experimento'] = df['Experimento'].astype(str).str.strip()


Edad column preprocessing

In [53]:
# Convertimos el string a fecha
df['Marca temporal'] = pd.to_datetime(df['Marca temporal'], dayfirst=True, errors='coerce')

def calculate_age(age_value, ref_ts):
    # Hay gente que puso la edad numerica solo
    age_num = pd.to_numeric(str(age_value).strip(), errors='coerce')
    if pd.notna(age_num):
        return int(age_num)

   # si puso fecha, la calculamos y comparamos con la fecha de referencia del csv (ref_ts)
   # Esto genera un warning pero da igual porq no es nada que afecte el codigo
    birth_ts = pd.to_datetime(str(age_value).strip(), dayfirst=True, errors='coerce') 
    if pd.notna(birth_ts) and pd.notna(ref_ts):
        # Edad = Año actual - Año nacimiento y si no ha cumplido este año se le resta 1 mñas
        years = ref_ts.year - birth_ts.year - ((ref_ts.month, ref_ts.day) < (birth_ts.month, birth_ts.day))
        return int(years)

# Llamamos a la funcion
df['Edad'] = [calculate_age(v, t) for v, t in zip(df['Edad'], df['Marca temporal'])]



/tmp/ipykernel_15672/780635987.py:12: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  birth_ts = pd.to_datetime(str(age_value).strip(), dayfirst=True, errors='coerce')


In [54]:
patata_questions = {
    '  ¿De qué genero es la patata?  ': 'Solanum',
    '¿Qué persona de origen Francés expandió el uso gastronómico de la patata? ': 'Antoine Parmentier',
    '¿Qué funciones tenía inicialmente la patata? ': 'De jardín',
    '¿En las cercanías de que lago fue domesticada la patata? ': 'Titiaca'
}

# Precompute experiment masks/totals once
exp_masks = {exp: df['Experimento'].eq(exp) for exp in ['1', '2']} # Diccionario que dice si es modo claro u oscuroi
exp_totals = {exp: int(mask.sum()) for exp, mask in exp_masks.items()} # Pa contar cuantos hay en cada exp

summary_rows = []
for question, correct_answer in patata_questions.items():
    q_answer_col = f"{question}__is_correct" # Aqui guardaremos si es corecta o no
    answers_clean = df[question].astype(str).str.strip()
    df[q_answer_col] = answers_clean.eq(correct_answer) # miramos si es correcta

    total_answered = int(answers_clean.notna().sum())
    total_correct = int(df[q_answer_col].sum())

    exp1_correct = int(df.loc[exp_masks['1'], q_answer_col].sum()) # Correctas
    exp2_correct = int(df.loc[exp_masks['2'], q_answer_col].sum())
    
    # % = correctas / total en ese modo * 100 
    pct_exp1 = (exp1_correct / exp_totals['1']) * 100
    pct_exp2 = (exp2_correct / exp_totals['2']) * 100

    summary_rows.append({
        'Pregunta': question.strip(),
        'Respuesta correcta': correct_answer,
        'Correctas': total_correct,
        'Incorrectas': total_answered - total_correct,
        'Pct correcta Exp 1': pct_exp1,
        'Pct correcta Exp 2': pct_exp2
    })

results_patata = pd.DataFrame(summary_rows)
results_patata

,Pregunta,Respuesta correcta,Correctas,Incorrectas,Pct correcta Exp 1,Pct correcta Exp 2
0,¿De qué genero es la patata?,Solanum,15,19,29.411765,58.823529
1,¿Qué persona de origen Francés expandió el uso...,Antoine Parmentier,21,13,41.176471,82.352941
2,¿Qué funciones tenía inicialmente la patata?,De jardín,20,14,70.588235,47.058824
3,¿En las cercanías de que lago fue domesticada ...,Titiaca,24,10,64.705882,76.470588


In [55]:
mochila_questions = {
    '¿Cumple el producto las restricciones de Ryanair?  ': 'Sí',
    '¿Cuánto es la capacidad del producto?  ': '20L',
    '¿De qué color es el producto?  ': 'Negro',
    '¿Para qué sirve el compartimento aislado de la parte principal?': 'Para objetos húmedos'
}

# Precompute experiment masks/totals once
exp_masks = {exp: df['Experimento'].eq(exp) for exp in ['1', '2']} # Diccionario que dice si es modo claro u oscuroi
exp_totals = {exp: int(mask.sum()) for exp, mask in exp_masks.items()} # Pa contar cuantos hay en cada exp

summary_rows = []
for question, correct_answer in mochila_questions.items():
    q_answer_col = f"{question}__is_correct" # Aqui guardaremos si es corecta o no
    answers_clean = df[question].astype(str).str.strip()
    df[q_answer_col] = answers_clean.eq(correct_answer) # miramos si es correcta

    total_answered = int(answers_clean.notna().sum())
    total_correct = int(df[q_answer_col].sum())

    exp1_correct = int(df.loc[exp_masks['1'], q_answer_col].sum()) # Correctas
    exp2_correct = int(df.loc[exp_masks['2'], q_answer_col].sum())
    
    # % = correctas / total en ese modo * 100 
    pct_exp1 = (exp1_correct / exp_totals['1']) * 100
    pct_exp2 = (exp2_correct / exp_totals['2']) * 100

    summary_rows.append({
        'Pregunta': question.strip(),
        'Respuesta correcta': correct_answer,
        'Correctas': total_correct,
        'Incorrectas': total_answered - total_correct,
        'Pct correcta Exp 1': pct_exp1,
        'Pct correcta Exp 2': pct_exp2
    })

results_mochila = pd.DataFrame(summary_rows)
results_mochila

,Pregunta,Respuesta correcta,Correctas,Incorrectas,Pct correcta Exp 1,Pct correcta Exp 2
0,¿Cumple el producto las restricciones de Ryanair?,Sí,34,0,100.000000,100.000000
1,¿Cuánto es la capacidad del producto?,20L,18,16,47.058824,58.823529
2,¿De qué color es el producto?,Negro,28,6,88.235294,76.470588
3,¿Para qué sirve el compartimento aislado de la...,Para objetos húmedos,21,13,64.705882,58.823529


In [56]:
df = pd.read_csv('raw/Encuesta TFG.csv')



df = df.drop_duplicates(subset=['Número de participante'], keep='first').copy()
df['Experimento'] = df['Experimento'].astype(str).str.strip()


preference_cols = list(df.columns[df.columns.get_loc('Tipo de visión'):])

# Seleccionamos las preguntas con cada experimento
df_pref = df[['Experimento'] + preference_cols].copy()
claro = df_pref[df_pref['Experimento'] == '1'].copy()
oscuro = df_pref[df_pref['Experimento'] == '2'].copy()

# Hay que mirar si son categoricas o numericas y las separamos
numeric_pref_cols = []
for col in preference_cols:
    as_num = pd.to_numeric(df_pref[col], errors='coerce')
    if as_num.notna().sum() > 0:
        numeric_pref_cols.append(col)

categorical_pref_cols = [c for c in preference_cols if c not in numeric_pref_cols]

# Cogemos las numericas
claro_num = claro[numeric_pref_cols].apply(pd.to_numeric, errors='coerce')
oscuro_num = oscuro[numeric_pref_cols].apply(pd.to_numeric, errors='coerce')

# Cogemos las descriptivas y calculamos el minimo, maximo , media y stf
desc_claro = claro_num.agg(['min', 'max', 'mean', 'std']).T
desc_oscuro = oscuro_num.agg(['min', 'max', 'mean', 'std']).T

# Juntamos el df
descriptive_by_experiment = pd.concat({
    'Experiment 1': desc_claro,
    'Experiment 2': desc_oscuro
}, axis=1)

print('Descriptive statistics by experiment (numeric preference questions):')
display(descriptive_by_experiment)

# Normality + test selection per numeric question
test_rows = []
for col in numeric_pref_cols:
    # Primero shapiro-wilk
    w1,p1 = stats.shapiro(claro_num[col])
    normal_claro = p1 > 0.05
    
    w2,p2 = stats.shapiro(oscuro_num[col])
    normal_oscuro = p2 > 0.05
    print(f"Probando normalidad para {col}:")
    print(f"Shapiro-Wilk Exp 1: W={w1:.4f}, p={p1:.6f} (Normal: {normal_claro})")
    print(f"Shapiro-Wilk Exp 2: W={w2:.4f}, p={p2:.6f} (Normal: {normal_oscuro})")
    if normal_claro and normal_oscuro:
        
        res = stats.ttest_ind(claro_num[col], oscuro_num[col], equal_var=False)
        print(f"Usando T-Test para {col} ya que ambos experimentos siguen distribución normal: t={res.statistic:.4f}, p={res.pvalue:.6f}")

    else:
        res = stats.mannwhitneyu(claro_num[col], oscuro_num[col], alternative='two-sided')
        print(f"Usando Mann-Whitney U para {col} ya que al menos un experimento no sigue distribución normal: U={res.statistic:.4f}, p={res.pvalue:.6f}")



# Mini analysis de los datos categoricos usando dataframe
if categorical_pref_cols:
    for col in categorical_pref_cols:
        display(pd.crosstab(df_pref['Experimento'], df_pref[col], margins=True))

Descriptive statistics by experiment (numeric preference questions):


Experiment 1       \
                                                            min  max   
En general, ¿Con qué frecuencia usas modo claro?            1.0  5.0   
En general, ¿Con qué frecuencia usas modo oscuro?           1.0  5.0   
Es fácil leer texto con la fuente y color usado             3.0  5.0   
Los colores usados en estas webs son atractivos             2.0  5.0   
En general, estoy satisfecho/a con la interfaz              2.0  5.0   
La combinación de colores hizo más fácil leer e...          3.0  5.0   
La combinación de colores hizo más fácil aprend...          2.0  5.0   
La combinación de colores me pareció placentera...          1.0  5.0   
La combinación de colores me pareció mostrar pr...          3.0  5.0   

                                                                        \
                                                        mean       std   
En general, ¿Con qué frecuencia usas modo claro?    3.294118  1.358524   
En general, ¿Con qué frecuencia usas modo oscuro?   2.823529  1.509772   
Es fácil leer texto con la fuente y color usado     4.000000  0.707107   
Los colores usados en estas webs son atractivos     3.705882  0.848875   
En general, estoy satisfecho/a con la interfaz      3.764706  0.752447   
La combinación de colores hizo más fácil leer e...  4.000000  0.866025   
La combinación de colores hizo más fácil aprend...  3.470588  0.943242   
La combinación de colores me pareció placentera...  3.411765  1.371989   
La combinación de colores me pareció mostrar pr...  4.117647  0.781213   

                                                   Experiment 2       \
                                                            min  max   
En general, ¿Con qué frecuencia usas modo claro?            1.0  5.0   
En general, ¿Con qué frecuencia usas modo oscuro?           1.0  5.0   
Es fácil leer texto con la fuente y color usado             2.0  5.0   
Los colores usados en estas webs son atractivos             2.0  5.0   
En general, estoy satisfecho/a con la interfaz              2.0  5.0   
La combinación de colores hizo más fácil leer e...          2.0  5.0   
La combinación de colores hizo más fácil aprend...          2.0  5.0   
La combinación de colores me pareció placentera...          1.0  5.0   
La combinación de colores me pareció mostrar pr...          3.0  5.0   

                                                                        
                                                        mean       std  
En general, ¿Con qué frecuencia usas modo claro?    3.647059  1.411612  
En general, ¿Con qué frecuencia usas modo oscuro?   2.764706  1.480262  
Es fácil leer texto con la fuente y color usado     4.235294  0.903425  
Los colores usados en estas webs son atractivos     3.647059  0.785905  
En general, estoy satisfecho/a con la interfaz      3.882353  0.927520  
La combinación de colores hizo más fácil leer e...  3.588235  0.870260  
La combinación de colores hizo más fácil aprend...  3.470588  1.067570  
La combinación de colores me pareció placentera...  3.352941  0.996317  
La combinación de colores me pareció mostrar pr...  4.411765  0.712287

Probando normalidad para En general, ¿Con qué frecuencia usas modo claro?:
Shapiro-Wilk Exp 1: W=0.8584, p=0.014450 (Normal: False)
Shapiro-Wilk Exp 2: W=0.8420, p=0.008111 (Normal: False)
Usando Mann-Whitney U para En general, ¿Con qué frecuencia usas modo claro? ya que al menos un experimento no sigue distribución normal: U=122.5000, p=0.443897
Probando normalidad para En general, ¿Con qué frecuencia usas modo oscuro?:
Shapiro-Wilk Exp 1: W=0.8647, p=0.018117 (Normal: False)
Shapiro-Wilk Exp 2: W=0.8764, p=0.027799 (Normal: False)
Usando Mann-Whitney U para En general, ¿Con qué frecuencia usas modo oscuro? ya que al menos un experimento no sigue distribución normal: U=147.0000, p=0.943579
Probando normalidad para Es fácil leer texto con la fuente y color usado:
Shapiro-Wilk Exp 1: W=0.8147, p=0.003247 (Normal: False)
Shapiro-Wilk Exp 2: W=0.7956, p=0.001766 (Normal: False)
Usando Mann-Whitney U para Es fácil leer texto con la fuente y color usado ya que al menos un experimento no sig

Tipo de visión,Con Corrección (Uso de gafas),Normal (Sin necesidad de corrección),Tengo un poco de miopia pero en el experimento no use gafas,gafas para leer y trabajar,myopia y astigmatismo,All
Experimento,,,,,,
1,9,7,0,0,1,17
2,9,6,1,1,0,17
All,18,13,1,1,1,34


¿Qué modo prefieres para el uso de Wikis?,Claro,Oscuro,All
Experimento,,,
1,11,6,17
2,9,8,17
All,20,14,34


¿Qué modo prefieres para el uso de Webs de comercio?,Claro,Oscuro,All
Experimento,,,
1,11,6,17
2,12,5,17
All,23,11,34
